# Clothing Classifier — Training

This project is about classifying 15 types of clothing, such as: blazers, jeans, hoodies, skirts, and others from a single photo. The model learns from 500 (400 train and 100 test/validation data) image example per category. The model we use here (timm) uses transfer learning, which basically means that the model has been trained on millions of everyday images and "knows" how to see edges, textures, and shapes, then retrain just the final part so it specializes in clothing. Kinda like hiring someone who knows how to identify the characteristics mentioned before and have them specializing in identifying garments.

## 1. Imports

In [ ]:
# Standard Python Libraries
import os
from collections import Counter
import random

# Numerical Data Analysis
import numpy as np

# PyTorch (the deep-learning engine)
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset

# torchvision (images for PyTorch)
from torchvision import datasets

# scikit-learn (classic ML utilities)
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Progress bar for loops
from tqdm import tqdm

# Models from the timm library (PyTorch Image Models)
import timm
from timm.data import resolve_data_config, create_transform

c:\Users\daru1\OneDrive\Desktop\Portfolio\clothing-classifier\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Here are the explanation for each modules:
1. Standard Python Libraries
- **os** is operating system helpers, which will be used to create the checkpoints folder (os.makedirs) and build file paths (os.path.join).
- **Counter** from **collections** is a module that primary used to count hashable objects. In this notebook, this is used to count the number of clothing labels in our dataset
- **random** is a standard python library that randomly generates number. Here we used it for seed generation so that the results are reproducible

2. Numerical/data analysis (**Numpy**)
- Here we only used **numpy**, which is the standard library for numerical arrays and math. The use is mainly np.random.seed which as mentioned previously, to fix a seed that can make the model be reproducible

3. PyTorch deep-learning engine
- **torch** is the core library that consists of tensors (the arrays the model works on) and all the GPU/CPU machinery
- **torch.nn(nn)** is the neural-network building blocks that consists of layers and loss functions. What we use below is nn.CrossEntropyLoss()
- **DataLoader, and Subset** from **torch.utils.data**: DataLoader feeds images to the model in batches, and Subset picks a slice of a dataset that carves into train and validation parts

4. images for PyTorch (**torchvision**)
- **datasets** is a ready-made dataset loaders. We use datasets.ImageFolder that reads images from one-folder-per-class layout

5. Scikit-Learn (classic ML tools)
- **train_test_split** splits the data into training and validation sets (we use it with stratification to keep classes balanced across the split)
- **classification_report, and confusion_matrix** are evaluation tools that we use for each class to evaluate their precision, recall, and F1. We also use it to visualize the confusion grid below

6. Progress+models
- **tqdm** draws the progress bars during training loops
- **timm** is the library that has pre-trained image models (we use ConvNeXt-Tiny)
- **resolve_data_config, and create_transform** from **timm.data** are used to figure out the exact preprocessing the model expects, and build the image transforms to match it so the inputs are normalized the same way the model was trained

## 2. Configuration

In [ ]:
# --- data ---
DATA_DIR   = r"C:\Users\daru1\OneDrive\Desktop\Portfolio\Clothes_Dataset"
CKPT_DIR   = "checkpoints" # this will act as the checkpoints for the best model trained

# --- model ---
MODEL_NAME = "convnext_tiny" # timm backbone to fine-tune (previously we used resnet18)

# --- training ---
BATCH_SIZE      = 32 # Image processed per epoch step (higher = faster/steadier, but need more memory)
VAL_SPLIT       = 0.2 # split the validation into 20% and the training 80%
SEED            = 42 
FREEZE_BACKBONE = True       # True = train only the head (fast on CPU), False = fine-tune whole network
EPOCHS          = 3          # full passes over the training data (3 = quick CPU test; 15-30 on GPU)
LR              = 1e-3 if FREEZE_BACKBONE else 1e-4 # higher Learning Rate when we train head-only, so that it is gentler when we fine-tune all layers
WEIGHT_DECAY    = 1e-4 # regularization that penalizes large weights to curb overfitting

device = "cuda" if torch.cuda.is_available() else "cpu" # check if NVIDIA GPU is available
print("device:", device)

device: cpu


This part is the control panel for the whole notebook by acting as the setting in one place, so we can tune training here rather than editing the codes that are scattered across the cells.

## 3. Dataset & class inspection
Confirm the folder layout: one subfolder per class, balanced counts.

In [4]:
base = datasets.ImageFolder(DATA_DIR)
print("classes found:", len(base.classes))
print("total images :", len(base))
print()
counts = Counter(base.targets)
for i, cls in enumerate(base.classes):
    print(f"{cls:<25} {counts[i]}")

classes found: 15
total images : 7500

Blazer                    500
Celana_Panjang            500
Celana_Pendek             500
Gaun                      500
Hoodie                    500
Jaket                     500
Jaket_Denim               500
Jaket_Olahraga            500
Jeans                     500
Kaos                      500
Kemeja                    500
Mantel                    500
Polo                      500
Rok                       500
Sweter                    500


This part is where we inspect the dataset and classes. We found out that the classes and data are well-balanced. Which means there will be no over or undersampling needed and we can continue below

## 4. Model & transforms
Transforms are pulled from the model so normalization always matches the backbone.

In [ ]:
num_classes = len(base.classes) # counts how many clothing categories/classes
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=num_classes) # pretrained on ImageNet, and will be fine-tuned later

cfg = resolve_data_config({}, model=model) # asks the mmodel what preprocessing it has been trained with in the result below
train_tf = create_transform(**cfg, is_training=True)    # adds augmentations for training (random crop, flip, etc.)
val_tf   = create_transform(**cfg, is_training=False)   # clean validation (center crop, no flip, etc.)

model = model.to(device) # move model onto the CPU
print("input config:", cfg)

input config: {'input_size': (3, 224, 224), 'interpolation': 'bicubic', 'mean': (0.485, 0.456, 0.406), 'std': (0.229, 0.224, 0.225), 'crop_pct': 0.95, 'crop_mode': 'center'}


Here from the result we see the previous preprocessing the model has been trained with are:
| Field | Details| 
| :---     | :---    | 
| 'input_size': (3, 224, 224)     | Every image is fed to the model as 3 color channels (RGB) × 224 × 224 pixels. So all inputs get resized to 224×224. | 
|   interpolation': 'bicubic'  | bicubic is a resampling method used when resizing. Bicubic in particular gives smooth results (better than plain "nearest")|
 | mean / std   | This is per-channel normalization values. Each pixel is transofrmed as (pixel-mean)/std. These specific numbers are the standard ImageNet statistics, which will be used because the model was pre-trained on ImageNet, so the emages must be scaled the same way.   | 
 |crop_pct': 0.95    | This acts as evaluation, the image is resized a bit larger, then the central 95% is cropped to 224 x 224, which is a light trim that focuses on the middle of the image   | 
 crop_mode': 'center'     | That crop is taken from the center as opposed to random, so it's more deterministic. This is more appropriate for eval data   | 

## 5. Train / validation split & DataLoaders
Stratified 80/20 split — keeps 100 of each class in validation.

In [6]:
train_idx, val_idx = train_test_split(
    range(len(base.targets)),
    test_size=VAL_SPLIT,
    stratify=base.targets,
    random_state=SEED,
)

train_ds = Subset(datasets.ImageFolder(DATA_DIR, transform=train_tf), train_idx)
val_ds   = Subset(datasets.ImageFolder(DATA_DIR, transform=val_tf),   val_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("train:", len(train_ds), " val:", len(val_ds))   # expect 6000 / 1500

train: 6000  val: 1500


Here we have 400 training data and 100 eval data for every label

## 6. Sanity-check one batch
Catch any image-reading problem in seconds, before a full epoch.

In [7]:
imgs, labels = next(iter(train_loader))
print("batch images:", imgs.shape)   # expect [32, 3, 224, 224]
print("batch labels:", labels[:8])

batch images: torch.Size([32, 3, 224, 224])
batch labels: tensor([ 9, 12,  1,  6,  6, 10, 11,  6])


The batch images are as we expected, consists of 3 color channels of RGB with the pixel width x height to be 224 x 224.

The batch labels are the first 8 labels shown.

## 7. Training setup
This is the part where we decide what will actually be trained. TThis is the part where we decide what will actually be trained: the freeze policy, the loss, and the optimizer.

In [24]:
# freeze everything, then re-enable just the classifier head
if FREEZE_BACKBONE:
    for p in model.parameters():
        p.requires_grad = False
    for p in model.get_classifier().parameters():
        p.requires_grad = True

criterion = nn.CrossEntropyLoss()
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)

print(f"trainable params: {sum(p.numel() for p in params):,}")

trainable params: 11,535


The freeze policy:
- **requires_grad** is a flag on each parameter that tells PyTorch "compute gradients for this and update it during training." Setting it False freezes the parameter (it won't change); True makes it trainable.

The loss function:
- ``criterion = nn.CrossEntropyLoss()`` is a loss function for multi-class classification, which measures how wrong the predicted class probabilities are versus the true labels.

Parameters and optimizers:
- **params** collects only the parameter still marked trainable, which is the head in this case
- **AdamW** is the optimizer, which happens to be an algorithm that adjust weight to reduce the loss. We hand the algorithm only the trainable parameters, so it updates the head and ignores the frozen backbone, using the learning rate **LR** and weight decay **WEIGHT_DECAY** from the config

**What is trainable params: 11,535 means?**

It's the parameter numbers that happens to be the one trained out of every ConvNeXt-Tiny's 28 million total parameters. This number is calculated from the ConvNeXt-Tiny's 768 feature dimension with our 15 classes:
$$
768 \times 15(weights)+15(biases)=11,520+15=11,535
$$
Above is the formula that decides the number of paramters trained from ConvNeXt-Tiny

## 8. Training loop

Below is the core loop that runs one full epoch over the dataset that handles both train and validation data via the **train** flag. This function returns two numbers: the average loss and the accuracy for that pass

In [ ]:
def run_epoch(loader, train):
    model.train() if train else model.eval() # set the model to training or evaluation mode
    total_loss, correct, seen = 0.0, 0, 0 # initialize metrics
    with torch.set_grad_enabled(train):
        for imgs, labels in tqdm(loader, leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            if train:
                optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            correct += (out.argmax(1) == labels).sum().item()
            seen += imgs.size(0)
    return total_loss / seen, correct / seen

### 8.1 Phase 1 Training
This is the first phase, which acts as the frozen-head wam-up because **FREEZE_BACKBONE=True** from the config only train the 11,535 parameters here.

In [26]:
best_acc = 0.0
os.makedirs(CKPT_DIR, exist_ok=True)

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader, train=False)
    print(f"epoch {epoch}/{EPOCHS} | "
          f"train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
          f"val loss {va_loss:.3f} acc {va_acc:.3f}")

    if va_acc > best_acc:
        best_acc = va_acc
        torch.save({
            "model_state": model.state_dict(),
            "model_name": MODEL_NAME,
            "classes": base.classes,
            "class_to_idx": base.class_to_idx,
            "input_size": cfg["input_size"],
        }, os.path.join(CKPT_DIR, "best.pt"))
        print(f"  saved best (val acc {best_acc:.3f})")

print("done. best val acc:", round(best_acc, 3))

epoch 1/3 | train loss 1.183 acc 0.615 | val loss 0.751 acc 0.746
  saved best (val acc 0.746)


epoch 2/3 | train loss 0.918 acc 0.698 | val loss 0.717 acc 0.756
  saved best (val acc 0.756)


epoch 3/3 | train loss 0.866 acc 0.705 | val loss 0.701 acc 0.757
  saved best (val acc 0.757)
done. best val acc: 0.757


>First phase and we already have the validation accuracy of 75.7%

### 8.2 Phase 2 Training
This is the phase where we unfreeze the model, which happens to be the rest 28 million training parameters in our model.

In [27]:
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

This Re-seeds all three random generatoes (PyTorch, NumPy, Python) to the fixed value. This makes the fine-tuning run  reproducible because the augmentation randomness, shuffling, etc. will come out the same every time we run it.

In [28]:
UNFREEZE_EPOCHS = 5
FT_LR = 1e-4   

We haven't unfreeze anything here, just setting up the number of runs (epoch) in this phase.

The learning rate here is 10 times smaller compared to the first phase, so we update the pre-trained backbone. Here we take gentle stpes so we refine the knowledge instead of destroying it. Big steps (learning rate) on the whole network would wreck the valuable ImageNet features.

In [29]:
for p in model.parameters():
    p.requires_grad = True

This is the unfreeze itself, since it sets every parameter trainable, which is the 28 million ones and not just the 11,535 head ones.

In [30]:
# fresh optimizer over ALL params (the old one only knew about the head)
optimizer = torch.optim.AdamW(model.parameters(), lr=FT_LR, weight_decay=1e-4)

# cosine schedule: LR eases down toward 0 over the run for a cleaner finish
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=UNFREEZE_EPOCHS)

for epoch in range(1, UNFREEZE_EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader, train=False)
    scheduler.step()
    print(f"[finetune] epoch {epoch}/{UNFREEZE_EPOCHS} | "
          f"train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
          f"val loss {va_loss:.3f} acc {va_acc:.3f}")

    if va_acc > best_acc:
        best_acc = va_acc
        torch.save({
            "model_state": model.state_dict(),
            "model_name": MODEL_NAME,
            "classes": base.classes,
            "class_to_idx": base.class_to_idx,
            "input_size": cfg["input_size"],
        }, "checkpoints/best.pt")
        print(f"  saved best (val acc {best_acc:.3f})")

print("finetune done. best val acc:", round(best_acc, 3))

  0%|          | 0/188 [00:00<?, ?it/s]

[finetune] epoch 1/5 | train loss 1.135 acc 0.627 | val loss 0.727 acc 0.757


[finetune] epoch 2/5 | train loss 0.827 acc 0.720 | val loss 0.661 acc 0.769
  saved best (val acc 0.769)


[finetune] epoch 3/5 | train loss 0.646 acc 0.780 | val loss 0.665 acc 0.783
  saved best (val acc 0.783)


[finetune] epoch 4/5 | train loss 0.490 acc 0.831 | val loss 0.595 acc 0.799
  saved best (val acc 0.799)


[finetune] epoch 5/5 | train loss 0.359 acc 0.876 | val loss 0.550 acc 0.821
  saved best (val acc 0.821)
finetune done. best val acc: 0.821


This is the best score we had so far for the validation data, which is 82.1% 

Let's create the checkpoint model reloader:

In [ ]:
# load your 0.821 weights back into the fresh model
ckpt = torch.load("checkpoints/best.pt", map_location=device)
model.load_state_dict(ckpt["model_state"])
model.to(device)

# unfreeze everything for full fine-tuning
for p in model.parameters():
    p.requires_grad = True

criterion = nn.CrossEntropyLoss()   # run_epoch needs this
best_acc = 0.821                    # so later phase only overwrites best.pt on a real improvement
print("loaded checkpoint, all params unfrozen")

loaded checkpoint, all params unfrozen


## 9. Evaluation
Below are the classic classification report and confussion matrix:

In [13]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        out = model(imgs.to(device))
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=base.classes))
print(confusion_matrix(all_labels, all_preds))

                precision    recall  f1-score   support

        Blazer       0.81      0.73      0.77       100
Celana_Panjang       0.81      0.79      0.80       100
 Celana_Pendek       0.95      0.92      0.93       100
          Gaun       0.77      0.89      0.82       100
        Hoodie       0.88      0.93      0.90       100
         Jaket       0.67      0.67      0.67       100
   Jaket_Denim       0.86      0.92      0.89       100
Jaket_Olahraga       0.76      0.64      0.70       100
         Jeans       0.87      0.91      0.89       100
          Kaos       0.86      0.91      0.88       100
        Kemeja       0.78      0.69      0.73       100
        Mantel       0.84      0.79      0.81       100
          Polo       0.69      0.75      0.72       100
           Rok       0.86      0.81      0.84       100
        Sweter       0.85      0.90      0.87       100

      accuracy                           0.82      1500
     macro avg       0.82      0.82      0.82 

From the confussion matrix, we see that the labels that get falsely classified the most are:
1. The jacket group:
- **Jaket_Olahraga** $\to$ **Jaket** (20 of 100), so it means that a true sports jacket will most likely be categorized as a plain jacket one time in 5 trials
- **Jaket** $\to$ **Jaket_Olahraga** (10 of 100), $\to$ **Jaket_Denim** (9 of 100), and $\to$ **Hoodie** (5 of 100). Here we see that **Jaket** is the lowest scoring class overall (F1 of 10.67), bleeding into all its cousins
2. Tops:
- **Kemeja** $\to$ **Polo** (20 of 100), so a shirt can be mislabeled as polo one time out of five trials.
- **Polo** $\to$ **Kemeja** (12 of 100), so a Polo can also be mislabeled as a shirt too.
3. **Blazer** $\leftrightarrow$ **Mantel** (10 of 100). So, blazer and coat trade places symmetrically
4. **Rok** $\to$ **Gaun** (9 of 100). So, a skirt has a probability to be mislabeled as a dress 9 out of 100 trials

## 10. Model Explanation
Let's print the model here:

In [12]:
print(model)   # full layer tree

ConvNeXt(
  (stem): Sequential(
    (0): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
    (1): LayerNorm2d((96,), eps=1e-06, elementwise_affine=True, bias=True)
  )
  (stages): Sequential(
    (0): ConvNeXtStage(
      (downsample): Identity()
      (blocks): Sequential(
        (0): ConvNeXtBlock(
          (conv_dw): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (norm): LayerNorm((96,), eps=1e-06, elementwise_affine=True, bias=True)
          (mlp): Mlp(
            (fc1): Linear(in_features=96, out_features=384, bias=True)
            (act): GELU()
            (drop1): Dropout(p=0.0, inplace=False)
            (norm): Identity()
            (fc2): Linear(in_features=384, out_features=96, bias=True)
            (drop2): Dropout(p=0.0, inplace=False)
          )
          (shortcut): Identity()
          (drop_path): Identity()
        )
        (1): ConvNeXtBlock(
          (conv_dw): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), paddi

### Model Architecture: ConvNeXt-Tiny

The model processes an image through three parts: a **stem**, four **stages**, and a **classifier head**. As the image goes deeper, its spatial size shrinks while the number of channels (features) grows — trading resolution for richer representation.

#### Overall flow
| Part | Operation | Output (channels × H × W) |
|------|-----------|---------------------------|
| **Stem** | 4×4 conv, stride 4 (+ LayerNorm) | 96 × 56 × 56 |
| **Stage 0** | 3 blocks, no downsample | 96 × 56 × 56 |
| **Stage 1** | downsample (96→192) + 3 blocks | 192 × 28 × 28 |
| **Stage 2** | downsample (192→384) + 9 blocks | 384 × 14 × 14 |
| **Stage 3** | downsample (384→768) + 3 blocks | 768 × 7 × 7 |
| **Head** | global average pool → Linear(768 → 15) | 15 class scores |

Block layout is `[3, 3, 9, 3]` with channel widths `[96, 192, 384, 768]`
> This is the standard ConvNeXt-Tiny configuration.

#### The stem
`Conv2d(3, 96, kernel=4, stride=4)` chops the `3×224×224` image into non-overlapping **4×4 patches**, projecting each to 96 features. This "patchify" stem is borrowed from Vision Transformers.

#### The downsample (start of stages 1–3)
A `LayerNorm` + a `2×2 stride-2 conv` that **halves the height/width and doubles the channels**. This is what steps the feature map down from 56 → 28 → 14 → 7 while growing 96 → 192 → 384 → 768.

#### The repeating unit: `ConvNeXtBlock`
Every stage is built from these identical blocks:
- **`conv_dw`** — a **depthwise 7×7 convolution** (`groups = channels`, so each channel is filtered on its own). The large 7×7 kernel gives a wide *receptive field*, so each feature "sees" a big region of the image — good for shape and texture.
- **`norm`** — LayerNorm, stabilizes the activations.
- **`mlp`** — an **inverted bottleneck**: `fc1` expands the channels 4× (e.g. 96 → 384), a **GELU** activation adds non-linearity, then `fc2` projects back down (384 → 96). This mirrors a Transformer's feed-forward block.
- **`shortcut: Identity`** — a **residual connection**: the block adds its output back to its input, so it only learns a *change*. This is what lets deep networks train stably.
- **`drop_path: Identity`** — stochastic depth, disabled here (`p=0`, as are all the dropouts, since the model is in evaluation/inference mode).

#### The head: `NormMlpClassifierHead`
1. **`global_pool` (average)** collapses the final `768 × 7 × 7` grid into a single **768-number vector** — one summary value per channel.
2. **LayerNorm → flatten** produce the clean 768-length feature vector.
3. **`fc: Linear(768 → 15)`** maps those 768 features to the **15 clothing-class scores** — this is the only part retrained for our task (768 × 15 + 15 = 11,535 parameters).

#### Conclusion
A `3×224×224` image is patchified to 96 channels, passed through four stages that progressively deepen features (96→192→384→768) while shrinking the map (56→28→14→7), pooled into a single 768-value fingerprint, and mapped to 15 clothing classes — a "modernized ResNet" that borrows the patchify stem, large depthwise convolutions, inverted-bottleneck MLPs, GELU, and LayerNorm from Vision Transformers.